In [3]:
!pip install torch

In [4]:
!pip install sentence-transformers 

  Using cached sentence_transformers-6.2.0-py3-none-any.whl.metadata (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 14.7 MB/s  0:00:00


In [8]:
import time
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np

# ==========================================
# 1. 模拟真实数据（Mock 知识库）
# ==========================================
# 假设这是我们公司关于“数码产品售后与政策”的知识库文档
documents = [
    "苹果（Apple）手机 iPhone 16 的保修期为自购买之日起一年，主要针对非人为损坏的硬件故障。",
    "如何把苹果做成沙拉？首先将苹果洗净去皮，切成小块，加入沙拉酱搅拌均匀即可。",
    "如果您的 iPhone 手机屏幕因不小心摔碎，这属于人为损坏，不在一年免费保修范围内，需自费维修。",
    "华为（Huawei）Mate 系列手机提供全国联保，电池保修期同样为一年，老化严重可申请免费更换。",
    "苹果官方零售店（Apple Store）支持 14 天内无理由退换货，但商品外包装需保持完好。",
    "保修期内如果手机频繁死机、无法开机，可以前往官方授权服务点进行免费检测和维修。",
    "如何选购新鲜的红富士苹果？应选择果形端正、果皮有条纹红、果蒂新鲜的果实。",
    "苹果手机屏幕故障可以提供付费售后服务，全国联保",
    "如果您的 iPhone 手机屏幕因不小心摔碎，您又买了相应的售后保险服务，那可以申请免费更换。"
]

# ==========================================
# 2. 初始化模型（基于现代 BERT 变体的底座）
# ==========================================
print("正在加载模型（首次运行会自动下载）...")

# 【粗筛阶段】使用基于 MiniLM (BERT轻量变体) 的经典双塔 Embedding 模型
# 这个模型把一句话变成一个 384 维的向量
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# 【精排阶段】使用基于 BERT/RoBERTa 架构的交叉编码器 Rerank 模型
# 它负责把 Query 和 Document 拼接在一起进行 Token 级别的深度交互打分
rerank_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("模型加载完成！\n" + "="*50)

# ==========================================
# 3. 离线阶段：为知识库建立向量索引（Embedding）
# ==========================================
# 真实场景中，这一步是提前跑好并存入向量数据库（如 Milvus/FAISS）的
print("【离线步】正在将海量文本转化为高维语义向量...")
doc_embeddings = embedding_model.encode(documents, convert_to_numpy=True)
print(f"成功将 {len(documents)} 篇文档转化为向量，矩阵形状: {doc_embeddings.shape}")
print("="*50)


# ==========================================
# 4. 在线阶段：流水线协同作战（Pipeline）
# ==========================================
def rag_search_pipeline(query, top_k_recall=4, top_n_rerank=2):
    print(f"\n🔍 用户提问: \"{query}\"")
    
    # ------------------------------------------
    # 阶段一：Embedding 粗筛 (Recall)
    # ------------------------------------------
    start_time = time.time()
    
    # 1. 将用户问题向量化
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    
    # 2. 计算其余弦相似度（快如闪电，适合百万级数据）
    # cos_sim = (A · B) / (||A|| * ||B||)
    norm_query = query_embedding / np.linalg.norm(query_embedding)
    norm_docs = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)
    similarities = np.dot(norm_docs, norm_query)
    
    # 3. 捞出最相关的前 K 个候选者
    top_k_indices = np.argsort(similarities)[::-1][:top_k_recall]
    recalled_docs = [documents[idx] for idx in top_k_indices]
    
    print(f"⏱️ 阶段一【Embedding 粗筛】耗时: {(time.time() - start_time)*1000:.2f} ms")
    print(f"💡 粗筛捞出的前 {top_k_recall} 个候选文档：")
    for i, idx in enumerate(top_k_indices):
        print(f"   [{i+1}] (得分: {similarities[idx]:.4f}) {documents[idx]}")
        
    # ------------------------------------------
    # 阶段二：Rerank 精排 (Reranking)
    # ------------------------------------------
    start_time = time.time()
    
    # 1. 将 Query 和 粗筛出来的文档 两两配对拼接
    # 形如: [["用户问题", "候选文档1"], ["用户问题", "候选文档2"], ...]
    pair_inputs = [[query, doc] for doc in recalled_docs]
    
    # 2. 让 Rerank 模型（基于 BERT 交叉编码）进行深度上下文交互打分
    rerank_scores = rerank_model.predict(pair_inputs)
    
    # 3. 根据精排得分重新排序
    rerank_indices = np.argsort(rerank_scores)[::-1][:top_n_rerank]
    
    print(f"\n⏱️ 阶段二【Rerank 精排】耗时: {(time.time() - start_time)*1000:.2f} ms")
    print(f"🏆 最终精排胜出、送往大模型的前 {top_n_rerank} 个黄金答案：")
    for i, idx in enumerate(rerank_indices):
        print(f"   🥇 Top-{i+1} (精排得分: {rerank_scores[idx]:.4f}) -> {recalled_docs[idx]}")


# ==========================================
# 5. 测试运行
# ==========================================
# 测试用例：用户问手机坏了能不能免费修
# 观察重点：Embedding 可能会把“水果沙拉”或“红富士苹果”因为字面有“苹果”也捞进来，
#         但 Rerank 能够敏锐识别逻辑，把真正讲“手机保修”的提到最前面。
rag_search_pipeline(query="我的苹果手机不小心摔坏了，能免费修理吗？", top_k_recall=4, top_n_rerank=2)


正在加载模型（首次运行会自动下载）...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

模型加载完成！
【离线步】正在将海量文本转化为高维语义向量...
成功将 9 篇文档转化为向量，矩阵形状: (9, 384)

🔍 用户提问: "我的苹果手机不小心摔坏了，能免费修理吗？"
⏱️ 阶段一【Embedding 粗筛】耗时: 4.50 ms
💡 粗筛捞出的前 4 个候选文档：
   [1] (得分: 0.7733) 如果您的 iPhone 手机屏幕因不小心摔碎，您又买了相应的售后保险服务，那可以申请免费更换。
   [2] (得分: 0.7599) 如果您的 iPhone 手机屏幕因不小心摔碎，这属于人为损坏，不在一年免费保修范围内，需自费维修。
   [3] (得分: 0.6409) 如何把苹果做成沙拉？首先将苹果洗净去皮，切成小块，加入沙拉酱搅拌均匀即可。
   [4] (得分: 0.4581) 如何选购新鲜的红富士苹果？应选择果形端正、果皮有条纹红、果蒂新鲜的果实。

⏱️ 阶段二【Rerank 精排】耗时: 5.48 ms
🏆 最终精排胜出、送往大模型的前 2 个黄金答案：
   🥇 Top-1 (精排得分: 7.6901) -> 如果您的 iPhone 手机屏幕因不小心摔碎，这属于人为损坏，不在一年免费保修范围内，需自费维修。
   🥇 Top-2 (精排得分: 7.4203) -> 如果您的 iPhone 手机屏幕因不小心摔碎，您又买了相应的售后保险服务，那可以申请免费更换。
